In [72]:
import json
import re
import requests
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")

In [73]:
VIDEO_IDS   = [25, 26, 27, 28]
OUTPUT_DIR  = Path("..", "Moment-Retrieval-and-Video-Summarization", "Descriptions")
OLLAMA_MODEL = "llama3.2"

# which VLM prompt's description to use as input to the LLM
VLM_PROMPT_TYPES = ["short", "medium", "long"]
VLM_FRAME_COUNTS = [30, 50]

LLM_PROMPTS = {
    "concise": """You are an expert video annotator. Below is a description of a video.
Understand the overall context and narrative of the video based on the description provided.
Extract between 4 and 10 salient events from this description, describe them in 1-2 concise sentences each, and list them in chronological order.

Format EXACTLY like this:
Salient event 1: [description]
Salient event 2: [description]
...
Only output the numbered list, nothing else.

Video description:
{vlm_description}""",

    "detailed": """You are an expert video annotator. Below is a description of a video.
Your task is to identify the most important and distinct events that capture the essence of the video.

Extract between 4 and 10 salient events. For each event:
- Write 1-2 concise sentences describing what happens
- Focus on actions, interactions, and key moments
- Be specific about who does what
- Do NOT reference the video itself ("the video shows", "the camera pans", etc.)

Format EXACTLY like this:
Salient event 1: [description]
Salient event 2: [description]
...
Only output the numbered list, nothing else.

Video description:
{vlm_description}""",
}

In [74]:
def run_llm(vlm_description, prompt_template, model=OLLAMA_MODEL):
    prompt = prompt_template.format(vlm_description=vlm_description)
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    raw = response.json()["response"]
    return {
        "raw_response": raw,
        "prompt":       prompt,
        "model":        model,
        "timestamp":    datetime.now(timezone.utc).isoformat(),
    }

In [75]:
def parse_salient_events(llm_output):
    events = []
    for line in llm_output.strip().splitlines():
        if not line.strip():
            continue
        if ":" in line and line.lower().startswith("salient event"):
            description = line.split(":", 1)[1].strip()
        else:
            description = re.sub(r"^\s*Salient event\s*\d+\s*[:\.\)\-]*\s*", "", line, flags=re.I)
            description = re.sub(r"^\s*\d+\s*[:\.\)\-]*\s*", "", description)
        if description:
            events.append(description)
    return events

In [76]:
def add_llm_result_to_json(video_id, prompt_type, result, parsed_events):
    out_file = OUTPUT_DIR / f"video_{video_id}.json"

    if out_file.exists():
        with open(out_file, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = {"video_id": video_id, "llm_results": []}

    if "llm_results" not in data:
        data["llm_results"] = []

    data["llm_results"].append({
        "prompt_type":    prompt_type,
        "prompt":         result["prompt"],
        "model":          result["model"],
        "vlm_source":     VLM_PROMPT_TYPE,
        "vlm_frame_count": VLM_FRAMES_COUNT,
        "timestamp":      result["timestamp"],
        "raw_response":   result["raw_response"],
        "parsed_events":  parsed_events,
    })

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"            Updated: {out_file}")

In [77]:
for video_id in VIDEO_IDS:
    out_file = OUTPUT_DIR / f"video_{video_id}.json"

    if not out_file.exists():
        print(f"Problem: video_{video_id}.json not found — run VLM pipeline first")
        continue

    with open(out_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # build set of already done combos once per video
    already_done = {
        (r["prompt_type"], r["model"], r.get("vlm_source"), r.get("vlm_frame_count"))
        for r in data.get("llm_results", [])
    }

    for vlm_prompt_type in VLM_PROMPT_TYPES:
        for vlm_frames_count in VLM_FRAME_COUNTS:

            # find the matching VLM description
            vlm_description = next(
                (r["response"] for r in data.get("vlm_results", [])
                 if r["prompt_type"] == vlm_prompt_type and r["num_frames"] == vlm_frames_count),
                None
            )
            if not vlm_description:
                print(f"  Skipping video_{video_id} | vlm={vlm_prompt_type}, "
                      f"frames={vlm_frames_count} — no VLM result found")
                continue

            print(f"\nProcessing video_{video_id} | vlm={vlm_prompt_type}, frames={vlm_frames_count}...")

            for prompt_type, prompt_template in LLM_PROMPTS.items():
                if (prompt_type, OLLAMA_MODEL, vlm_prompt_type, vlm_frames_count) in already_done:
                    print(f"        Skipping '{prompt_type}' — already done")
                    continue

                print(f"        Running '{prompt_type}' prompt...")
                result = run_llm(vlm_description, prompt_template)
                parsed = parse_salient_events(result["raw_response"])
                add_llm_result_to_json(video_id, prompt_type, result, parsed)
                already_done.add((prompt_type, OLLAMA_MODEL, vlm_prompt_type, vlm_frames_count))
                print(f"        Done — {len(parsed)} events extracted")

print("\n\nAll videos processed!")


Processing video_25 | vlm=short, frames=30...
        Skipping 'concise' — already done
        Skipping 'detailed' — already done

Processing video_25 | vlm=short, frames=50...
        Running 'concise' prompt...
            Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_25.json
        Done — 10 events extracted
        Running 'detailed' prompt...
            Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_25.json
        Done — 10 events extracted

Processing video_25 | vlm=medium, frames=30...
        Skipping 'concise' — already done
        Skipping 'detailed' — already done

Processing video_25 | vlm=medium, frames=50...
        Running 'concise' prompt...
            Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_25.json
        Done — 6 events extracted
        Running 'detailed' prompt...
            Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_25.json
        Done — 5 event